In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import quimb.tensor as qtn

In [ ]:
def random_clifford_t_block(circ, q1, q2, add_t, p_t=0.2):
    for q in [q1, q2]:
        r = np.random.randint(4)
        if r == 0:
            circ.h(q)
        elif r == 1:
            circ.s(q)
        elif r == 2:
            circ.h(q)
            circ.s(q)

    if np.random.rand() < 0.5:
        circ.cx(q1, q2)
    else:
        circ.cx(q2, q1)

    if add_t:
        if np.random.rand() < p_t:
            circ.t(q1)
        if np.random.rand() < p_t:
            circ.t(q2)


def build_circuit_2d_mps(Lx, Ly, d, add_t, p_t=0.2, max_bond=64, cutoff=1e-10):
    """Same 2D brickwork structure as build_circuit_2d, but built directly as a
    quimb CircuitMPS so the state is never materialized as a full 2^N vector."""
    N = Lx * Ly
    circ = qtn.CircuitMPS(N, max_bond=max_bond, cutoff=cutoff)

    def idx(x, y):
        return y * Lx + x

    for layer in range(d):
        direction = layer % 2
        offset = (layer // 2) % 2

        if direction == 0:
            for y in range(Ly):
                for x in range(offset, Lx - 1, 2):
                    random_clifford_t_block(circ, idx(x, y), idx(x + 1, y), add_t=add_t, p_t=p_t)
        else:
            for x in range(Lx):
                for y in range(offset, Ly - 1, 2):
                    random_clifford_t_block(circ, idx(x, y), idx(x, y + 1), add_t=add_t, p_t=p_t)

    return circ


def projected_cp_mps(circ, n_measured):
    """
    Equation (3) from the paper, evaluated exactly via MPS contraction:

        2^N * |<0|psi>|^4 / p_{0|psi}^2

    where the n_measured rightmost qubits are subsystem B (measured) and the
    rest are subsystem A (projected), N = total qubits. The 2^N prefactor (not
    2^n_projected) comes from twirling over BOTH subsystems to collapse the
    double sum in equation (2): a factor of 2^n_projected from the A-twirl and
    a factor of 2^n_measured from the B-twirl, giving 2^N total.

    Valid in expectation over an LU-invariant ensemble — average over many
    circuit instances to get the actual collision probability; a single
    instance is not meaningful on its own.

    Cost: O(N * chi^2) via MPS, independent of 2^N — this is what lets it scale
    past the ~25-qubit wall of the exact statevector method.

    Returns NaN if the all-zeros outcome on the measured subsystem has
    (numerically) zero probability for this instance. Aggregate with
    np.nanmean/np.nanstd so a single such instance doesn't wipe out the average.
    """
    N = circ.N

    amp = circ.amplitude('0' * N)
    p_all_zeros = abs(amp) ** 2

    measured_qubits = list(range(n_measured))
    marginal = circ.compute_marginal(where=measured_qubits)
    p_x0 = marginal[(0,) * n_measured]

    denom = p_x0 ** 2
    if not np.isfinite(denom) or denom <= 0:
        return np.nan

    return (2 ** N) * p_all_zeros ** 2 / denom

In [ ]:
# Vary N, fixed depth — MPS lets us go well past the statevector's N~25 wall
L_values = list(range(2, 9))   # N = 4 .. 64
d = 10
add_t = True
samples_per_n = 30
max_bond = 64
output_dir = "results/mps"
os.makedirs(output_dir, exist_ok=True)

avg_prob, sem_prob, total_qubits = [], [], []

for L in L_values:
    N = L * L
    n_measured = N // 2
    n_projected = N - n_measured
    vals = []

    for _ in range(samples_per_n):
        circ = build_circuit_2d_mps(L, L, d, add_t, p_t=0.15, max_bond=max_bond)
        vals.append(projected_cp_mps(circ, n_measured))

    n_valid = int(np.sum(~np.isnan(vals)))
    n_dropped = samples_per_n - n_valid
    print(f"L={L}, N={N}, n_measured={n_measured}, n_projected={n_projected}, dropped {n_dropped}/{samples_per_n} samples (p_x0=0)")

    avg_prob.append(np.nanmean(vals))
    sem_prob.append(np.nanstd(vals) / np.sqrt(max(n_valid, 1)))
    total_qubits.append(N)

    fig, ax = plt.subplots()
    ax.errorbar(total_qubits, avg_prob, yerr=sem_prob, marker='o', capsize=4, label=f"d={d}, MPS chi<={max_bond}")
    ax.set_xlabel("N (total qubits)")
    ax.set_ylabel(r"$2^N \, |\langle 0|\psi\rangle|^4 / p_{0|\psi}^2$  (eq. 3, MPS)")
    ax.set_yscale('log')
    ax.set_title(f"2D Brickwork: Projected Ensemble CP via MPS  (d={d}, varying N)")
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.savefig(os.path.join(output_dir, f"mps_cp_d{d}_upto_N{N}.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  saved mps_cp_d{d}_upto_N{N}.png")

plt.errorbar(total_qubits, avg_prob, yerr=sem_prob, marker='o', capsize=4, label=f"d={d}, MPS chi<={max_bond}")
plt.xlabel("N (total qubits)")
plt.ylabel(r"$2^N \, |\langle 0|\psi\rangle|^4 / p_{0|\psi}^2$  (eq. 3, MPS)")
plt.yscale('log')
plt.title(f"2D Brickwork: Projected Ensemble CP via MPS  (d={d}, varying N)")
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Vary depth, fixed N
L = 6           # N = 36 — infeasible for exact statevector, fine for MPS
d_values = list(range(1, 21))
add_t = True
samples_per_d = 30
max_bond = 64
output_dir = "results/mps"
os.makedirs(output_dir, exist_ok=True)

N = L * L
n_measured = N // 2
n_projected = N - n_measured

avg_prob, sem_prob, completed_depths = [], [], []

for d in d_values:
    vals = []

    for _ in range(samples_per_d):
        circ = build_circuit_2d_mps(L, L, d, add_t, p_t=0.15, max_bond=max_bond)
        vals.append(projected_cp_mps(circ, n_measured))

    n_valid = int(np.sum(~np.isnan(vals)))
    n_dropped = samples_per_d - n_valid
    print(f"d={d}, N={N}, n_measured={n_measured}, n_projected={n_projected}, dropped {n_dropped}/{samples_per_d} samples (p_x0=0)")

    avg_prob.append(np.nanmean(vals))
    sem_prob.append(np.nanstd(vals) / np.sqrt(max(n_valid, 1)))
    completed_depths.append(d)

    fig, ax = plt.subplots()
    ax.errorbar(completed_depths, avg_prob, yerr=sem_prob, marker='o', capsize=4, label=f"N={N}, MPS chi<={max_bond}")
    ax.set_xlabel("depth")
    ax.set_ylabel(r"$2^N \, |\langle 0|\psi\rangle|^4 / p_{0|\psi}^2$  (eq. 3, MPS)")
    ax.set_yscale('log')
    ax.set_title(f"2D Brickwork: Projected Ensemble CP via MPS  (N={N}, varying depth)")
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.savefig(os.path.join(output_dir, f"mps_cp_N{N}_upto_d{d}.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  saved mps_cp_N{N}_upto_d{d}.png")

plt.errorbar(completed_depths, avg_prob, yerr=sem_prob, marker='o', capsize=4, label=f"N={N}, MPS chi<={max_bond}")
plt.xlabel("depth")
plt.ylabel(r"$2^N \, |\langle 0|\psi\rangle|^4 / p_{0|\psi}^2$  (eq. 3, MPS)")
plt.yscale('log')
plt.title(f"2D Brickwork: Projected Ensemble CP via MPS  (N={N}, varying depth)")
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()